# 🐱🐶 Classification d'Images avec Data Augmentation (Chats vs Chiens)

- Prétraiter les données d'images pour un réseau de neurones convolutionnel (CNN)
- Appliquer des techniques d'augmentation de données pour améliorer la généralisation du modèle
- Construire et entraîner un CNN pour la classification binaire d'images
- Utiliser le dropout pour réduire le surapprentissage dans un réseau de neurones

- Un modèle de classification binaire d'images pour distinguer les chats et les chiens
- Une visualisation des métriques d'entraînement et de validation pour analyser les performances du modèle

---

**Note:** Ce notebook implémente toutes les 12 parties du projet de manière complète et structurée.

In [ ]:
# Partie 1: Téléchargement et chargement des données avec générateurs
# Télécharger le dataset depuis GitHub
!wget https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D5/Dogs%20vs%20Cats.zip -O cats_dogs.zip
!unzip -q cats_dogs.zip -d data
!mkdir -p data/cats_dogs

# Imports
import os, math, re, random
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42); tf.random.set_seed(42)

# Configuration globale
DATA_ROOT = Path("data/cats_dogs")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir = (DATA_ROOT / "test" / "test") if (DATA_ROOT / "test" / "test").exists() else (DATA_ROOT / "test")
IMG_HEIGHT, IMG_WIDTH = 48, 48  # Résolution réduite comme recommandé
batch_size = 32
seed = 1337

# Fonction pour construire DataFrame depuis dossiers
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f"No images found under {folder}")
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat", "cats"}:
                label = "cat"
            elif parent in {"dog", "dogs"}:
                label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name):
                    label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name):
                    label = "dog"
                else:
                    continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full = build_df_from_folder(test_dir, labeled=False)

# Train validation split
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed
)

# Partie 2: Inspection des données
print(f"\n📊 Inspection des données:")
print(f"Total images d'entraînement: {len(df_tr)}")
print(f"Total images de validation: {len(df_val)}")
print(f"\nDistribution des classes dans l'entraînement:")
print(df_tr['label'].value_counts())
print(f"\nÉquilibre des classes: {'Équilibré' if abs(df_tr['label'].value_counts().iloc[0] - df_tr['label'].value_counts().iloc[1]) < 500 else 'Déséquilibré'}")

# Visualisation d'exemples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
sample_imgs = df_tr.sample(10, random_state=42)
for idx, (ax, row) in enumerate(zip(axes.flat, sample_imgs.iterrows())):
    img = plt.imread(row[1]['filepath'])
    ax.imshow(img)
    ax.set_title(f"Label: {row[1]['label']}", fontsize=10)
    ax.axis('off')
plt.suptitle('Exemples d\'images d\'entraînement', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Générateurs avec augmentation
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode="binary",
    batch_size=batch_size, shuffle=True, seed=seed, validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode="binary",
    batch_size=batch_size, shuffle=False, validate_filenames=False
)
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col="filepath", y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode=None,
    batch_size=batch_size, shuffle=False, validate_filenames=False
)

print(f"\n✅ Générateurs créés:")
print(f"Train: {train_flow.samples}, Val: {val_flow.samples}, Test: {test_flow.samples}")
print(f"Class indices: {train_flow.class_indices}")

# Partie 3-4: Architecture CNN et optimisation
def create_cnn_model(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)):
    """
    Architecture CNN:
    - 3 blocs Conv2D + MaxPooling
    - Dropout pour régularisation
    - Dense layers avec sigmoid pour classification binaire
    """
    model = models.Sequential([
        # Bloc 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloc 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloc 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Flatten et Dense layers
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')  # Sortie binaire
    ])
    return model

# Créer et compiler le modèle avec augmentation
model_aug = create_cnn_model()
model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"\n📐 Architecture du modèle CNN:")
model_aug.summary()

# Partie 5: Entraînement avec callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

print(f"\n🚀 Début de l'entraînement avec augmentation de données...")
history_aug = model_aug.fit(
    train_flow,
    validation_data=val_flow,
    epochs=20,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Partie 6-7: Évaluation et visualisation
val_loss, val_acc = model_aug.evaluate(val_flow, verbose=0)
print(f"\n📊 Performance sur validation:")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

# Prédictions pour confusion matrix
val_flow.reset()
val_preds = (model_aug.predict(val_flow, verbose=0) > 0.5).astype(int).flatten()
val_true = val_flow.classes

# Confusion matrix et métriques
cm = confusion_matrix(val_true, val_preds)
print(f"\n📈 Matrice de confusion:")
print(cm)
print(f"\n📋 Rapport de classification:")
print(classification_report(val_true, val_preds, target_names=['cat', 'dog']))

# Visualisation des courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history_aug.history['loss'], label='Train Loss')
axes[0].plot(history_aug.history['val_loss'], label='Val Loss')
axes[0].set_title('Perte d\'Entraînement vs Validation (Avec Augmentation)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_aug.history['accuracy'], label='Train Accuracy')
axes[1].plot(history_aug.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Précision d\'Entraînement vs Validation (Avec Augmentation)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Partie 8: Baseline sans augmentation (comparaison)
train_gen_no_aug = ImageDataGenerator(rescale=1./255)
train_flow_no_aug = train_gen_no_aug.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode="binary",
    batch_size=batch_size, shuffle=True, seed=seed, validate_filenames=False
)

model_no_aug = create_cnn_model()
model_no_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"\n🔄 Entraînement du modèle baseline (SANS augmentation)...")
history_no_aug = model_no_aug.fit(
    train_flow_no_aug,
    validation_data=val_flow,
    epochs=20,
    callbacks=[early_stop],
    verbose=1
)

# Comparaison
val_loss_no_aug, val_acc_no_aug = model_no_aug.evaluate(val_flow, verbose=0)
print(f"\n📊 Comparaison des performances:")
print(f"Avec augmentation: Val Acc = {val_acc:.4f}, Val Loss = {val_loss:.4f}")
print(f"Sans augmentation: Val Acc = {val_acc_no_aug:.4f}, Val Loss = {val_loss_no_aug:.4f}")
print(f"Amélioration: {(val_acc - val_acc_no_aug) * 100:.2f}%")

# Partie 9-10: Prédictions test et sauvegarde
test_flow.reset()
test_probs = model_aug.predict(test_flow, verbose=0)
test_results = pd.DataFrame({
    'filepath': df_test_full['filepath'],
    'prob_dog': test_probs.flatten(),
    'pred_label': ['dog' if p > 0.5 else 'cat' for p in test_probs.flatten()]
})
test_results.to_csv('test_predictions.csv', index=False)
print(f"\n💾 Prédictions test sauvegardées dans 'test_predictions.csv'")
print(test_results.head(10))

# Sauvegarder le modèle
model_aug.save('cats_dogs_cnn_model.h5')
print(f"\n💾 Modèle sauvegardé: 'cats_dogs_cnn_model.h5'")

# Partie 11-12: Récapitulatif final
print(f"\n✅ PROJET TERMINÉ!")
print(f"\n📋 Livrables complétés:")
print(f"  ✓ Rapport de données avec distribution des classes")
print(f"  ✓ Visualisation d'exemples d'images")
print(f"  ✓ Architecture CNN définie et justifiée")
print(f"  ✓ Entraînement avec augmentation de données")
print(f"  ✓ Courbes d'apprentissage et d'évaluation")
print(f"  ✓ Matrice de confusion et métriques détaillées")
print(f"  ✓ Comparaison baseline vs augmentation")
print(f"  ✓ Prédictions sur ensemble de test (CSV)")
print(f"  ✓ Modèle sauvegardé pour réutilisation")
print(f"\n🎯 Performance finale: {val_acc*100:.2f}% accuracy sur validation")